# 03 · Final classifier: TTA + ensemble

**Goal:** pick the final classifier from the trained models, then measure it **once** on the test split.

1. Load the trained checkpoints from `02_classifier.ipynb` (EfficientNet-B0 baseline, EfficientNet-B3 strong), with and without the brain crop.
2. Score every option on **val**: each model alone, with test-time augmentation (TTA: original + flipped image), and the ensemble of each group (average of the B0 and B3 probabilities).
3. Pick the best option by val macro-F1.
4. Open the test split once (`RUN_TEST = True`) for the final number.

No training here, so a CPU is enough (a GPU is faster). Runs on Kaggle or locally.

**Inputs:** the brain tumor dataset, `splits.csv`, and the checkpoints listed in `MODELS` (local: `ml/models/`, Kaggle: upload them as a dataset). Missing checkpoints are skipped.

In [ ]:
import json
import os
from pathlib import Path

import albumentations as A
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import timm
import torch
from albumentations.pytorch import ToTensorV2
from PIL import Image
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score
from torch.utils.data import DataLoader, Dataset


def find_data_dir() -> Path:
    if "NEUROLENS_DATA" in os.environ:
        return Path(os.environ["NEUROLENS_DATA"])
    for p in Path("/kaggle/input").glob("**/Training"):
        return p.parent
    raise FileNotFoundError("Dataset not found. Attach masoudnickparvar/brain-tumor-mri-dataset.")


def find_file(name: str, local: Path) -> Path | None:
    for p in [*Path("/kaggle/input").glob(f"**/{name}"), local]:
        if p.exists():
            return p
    return None


def crop_brain(img: np.ndarray, margin: float = 0.03) -> np.ndarray:
    """Same crop as in 02_classifier: cut the slice to the head and pad it to a square."""
    gray = cv2.GaussianBlur(cv2.cvtColor(img, cv2.COLOR_RGB2GRAY), (5, 5), 0)
    mask = (gray > max(10, 0.08 * np.percentile(gray, 99.5))).astype(np.uint8) * 255
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    n, _, stats, _ = cv2.connectedComponentsWithStats(mask)
    if n < 2:
        return img
    x, y, w, h, area = stats[1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])]
    if area < 0.05 * mask.size:
        return img
    m = int(margin * max(w, h))
    x0, y0 = max(0, x - m), max(0, y - m)
    x1, y1 = min(img.shape[1], x + w + m), min(img.shape[0], y + h + m)
    crop = img[y0:y1, x0:x1]
    side = max(crop.shape[:2])
    out = np.zeros((side, side, 3), dtype=img.dtype)
    oy, ox = (side - crop.shape[0]) // 2, (side - crop.shape[1]) // 2
    out[oy:oy + crop.shape[0], ox:ox + crop.shape[1]] = crop
    return out


DATA_DIR = find_data_dir()
SPLITS_CSV = Path(os.environ.get("NEUROLENS_SPLITS", "")) if "NEUROLENS_SPLITS" in os.environ else find_file("splits.csv", Path("../data/splits.csv"))
MODELS_DIR = Path(os.environ.get("NEUROLENS_MODELS", "../models"))
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# name -> (timm backbone, image size, checkpoint file, brain crop)
MODELS = {
    "b0": ("efficientnet_b0", 224, "classifier_b0_baseline.pt", False),
    "b3": ("efficientnet_b3", 384, "classifier_b3_strong.pt", False),
    "b0_crop": ("efficientnet_b0", 224, "classifier_baseline_crop.pt", True),
    "b3_crop": ("efficientnet_b3", 384, "classifier_strong_crop.pt", True),
}
RUN_TEST = False  # open the test split only once, for the final pick
CLASSES = ["glioma", "meningioma", "notumor", "pituitary"]
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("data:", DATA_DIR, "| splits:", SPLITS_CSV, "| device:", device)

## 1. Data and models

In [ ]:
df = pd.read_csv(SPLITS_CSV)
df["y"] = df.label.map({c: i for i, c in enumerate(CLASSES)})
val_df = df[df.split == "val"].reset_index(drop=True)
test_df = df[df.split == "test"].reset_index(drop=True)


class MRIDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, size: int, crop: bool):
        self.paths = [DATA_DIR / p for p in frame.path]
        self.tf = A.Compose([A.Resize(size, size), A.Normalize(mean=MEAN, std=STD), ToTensorV2()])
        self.crop = crop

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = np.array(Image.open(self.paths[i]).convert("RGB"))
        return self.tf(image=crop_brain(img) if self.crop else img)["image"]


models = {}
for name, (arch, size, file, crop) in MODELS.items():
    ckpt = find_file(file, MODELS_DIR / file)
    if ckpt is None:
        print(f"skip {name}: {file} not found")
        continue
    m = timm.create_model(arch, pretrained=False, num_classes=len(CLASSES))
    m.load_state_dict(torch.load(ckpt, map_location="cpu"))
    models[name] = (m.eval().to(device), size, crop)
    print(f"loaded {name} from {ckpt}")
assert models, "No checkpoints found."

# Ensembles: average the models of one group (with or without the crop) when the group has 2+ models
GROUPS = {"ensemble+tta": [n for n in models if not models[n][2]],
          "ensemble_crop+tta": [n for n in models if models[n][2]]}
GROUPS = {k: v for k, v in GROUPS.items() if len(v) > 1}

## 2. Predict with and without TTA

In [ ]:
@torch.no_grad()
def predict(name: str, frame: pd.DataFrame, tta: bool) -> np.ndarray:
    """Class probabilities for every image. TTA averages the original and the left-right flipped image."""
    model, size, crop = models[name]
    loader = DataLoader(MRIDataset(frame, size, crop), batch_size=16, shuffle=False, num_workers=2 if device.type == "cuda" else 0)
    out = []
    for x in loader:
        x = x.to(device)
        p = model(x).float().softmax(1)
        if tta:
            p = (p + model(x.flip(3)).float().softmax(1)) / 2
        out.append(p.cpu())
    return torch.cat(out).numpy()


def score(probs: np.ndarray, ys: np.ndarray) -> dict:
    preds = probs.argmax(1)
    return dict(acc=round(accuracy_score(ys, preds), 4), macro_f1=round(f1_score(ys, preds, average="macro"), 4),
                errors=int((preds != ys).sum()))

## 3. Compare every option on val

In [ ]:
yv = val_df.y.to_numpy()
val_probs = {}
for name in models:
    val_probs[name] = predict(name, val_df, tta=False)
    val_probs[f"{name}+tta"] = predict(name, val_df, tta=True)
for group, names in GROUPS.items():
    val_probs[group] = np.mean([val_probs[f"{n}+tta"] for n in names], axis=0)

table = pd.DataFrame({k: score(p, yv) for k, p in val_probs.items()}).T.sort_values(["macro_f1", "acc"], ascending=False)
table.to_csv(OUT_DIR / "val_comparison.csv")
PICK = None  # None = best val score; or name an option to choose by hand (e.g. the crop models, whose heatmaps are more honest)
best = PICK or table.index[0]
print("best on val:", table.index[0], "| pick:", best)
table

## 4. Final pick on val

In [ ]:
def report(name: str, probs: np.ndarray, ys: np.ndarray, split: str) -> None:
    preds = probs.argmax(1)
    print(f"{split} · {name}")
    print(classification_report(ys, preds, labels=range(len(CLASSES)), target_names=CLASSES, digits=4, zero_division=0))
    ConfusionMatrixDisplay.from_predictions(ys, preds, labels=range(len(CLASSES)), display_labels=CLASSES, cmap="Blues", colorbar=False)
    plt.title(f"{split} confusion matrix · {name}")
    plt.savefig(OUT_DIR / f"confusion_final_{split}.png", dpi=80, bbox_inches="tight")
    plt.show()


report(best, val_probs[best], yv, "val")

## 5. Test (run once, for the final pick only)

In [ ]:
def probs_for(option: str, frame: pd.DataFrame) -> np.ndarray:
    if option in GROUPS:
        return np.mean([predict(n, frame, tta=True) for n in GROUPS[option]], axis=0)
    name, tta = option.replace("+tta", ""), option.endswith("+tta")
    return predict(name, frame, tta=tta)


test_metrics = None
if RUN_TEST:
    yt = test_df.y.to_numpy()
    test_probs = probs_for(best, test_df)
    test_metrics = score(test_probs, yt)
    print("TEST:", test_metrics)
    report(best, test_probs, yt, "test")
else:
    print("Test split closed. Set RUN_TEST = True once the pick on val is final.")

## 6. Save

In [ ]:
final = dict(pick=best, val=table.loc[best].to_dict(), test=test_metrics, models={k: MODELS[k] for k in models},
             n_val=len(val_df), n_test=len(test_df))
(OUT_DIR / "metrics_final.json").write_text(json.dumps(final, indent=2, default=int))
print(json.dumps(final, indent=2, default=int))